In [1]:
import pandas as pd
import numpy as np
from pgmpy.models import BayesianModel
from pgmpy.models import BayesianNetwork
from pgmpy.inference import VariableElimination, ApproxInference, BeliefPropagation
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.estimators import BayesianEstimator
from pgmpy.estimators import HillClimbSearch
from pgmpy.estimators import BDeuScore, K2Score, BicScore
from pgmpy.metrics import structure_score
from pgmpy.utils import get_example_model
from pgmpy.estimators import ScoreCache
from pgmpy.inference.CausalInference import CausalInference
import networkx as nx
import itertools
import math
import networkx as nx
import matplotlib.pyplot as plt

In [2]:
from same_decision_probability_calculation import *
from minimum_information_loss_partition import *
from utils import *

from monte_carlo_sdp import *

In [3]:
from pgmpy.utils import get_example_model

# Loading Models

In [4]:
import pandas as pd
from ucimlrepo import fetch_ucirepo
from pgmpy.models import NaiveBayes
from pgmpy.estimators import MaximumLikelihoodEstimator

# ── VOTING ──────────────────────────────────────────────────────────────────
voting = fetch_ucirepo(id=105)
df_voting = pd.concat([voting.data.features, voting.data.targets], axis=1)
df_voting.columns = [c.strip() for c in df_voting.columns]

# Replace '?' missing values — Naive Bayes needs complete data
df_voting = df_voting.replace('?', pd.NA).dropna()

# All values must be strings/categories for pgmpy
df_voting = df_voting.astype(str)

target_voting = 'Class'   # 'democrat' / 'republican'

voting_model = NaiveBayes()
voting_model.fit(df_voting, target_voting,
                 estimator=MaximumLikelihoodEstimator)

# ── CHESS ────────────────────────────────────────────────────────────────────
chess = fetch_ucirepo(id=22)
df_chess = pd.concat([chess.data.features, chess.data.targets], axis=1)
df_chess = df_chess.astype(str)

target_chess = 'skach' 

chess_model = NaiveBayes()
chess_model.fit(df_chess, target_chess,
                estimator=MaximumLikelihoodEstimator)

In [5]:
# cast models to pgmpy BayesianNetwork for compatibility with our code
voting_model = BayesianNetwork(voting_model.edges())
chess_model = BayesianNetwork(chess_model.edges())

# fit
voting_model.fit(df_voting, estimator=MaximumLikelihoodEstimator)
chess_model.fit(df_chess, estimator=MaximumLikelihoodEstimator)

In [6]:
# find binary variables in chess df
binary_vars_chess = [col for col in df_chess.columns if df_chess[col].nunique() == 2]
print(f"Binary variables in Chess dataset: {binary_vars_chess}")

Binary variables in Chess dataset: ['bkblk', 'bknwy', 'bkon8', 'bkona', 'bkspr', 'bkxbq', 'bkxcr', 'bkxwp', 'blxwp', 'bxqsq', 'cntxt', 'dsopp', 'dwipd', 'katri', 'mulch', 'qxmsq', 'r2ar8', 'reskd', 'reskr', 'rimmx', 'rkxwp', 'rxmsq', 'simpl', 'skach', 'skewr', 'skrxp', 'spcop', 'stlmt', 'thrsk', 'wkcti', 'wkna8', 'wknck', 'wkovl', 'wkpos', 'wtoeg']


In [7]:
child_model = get_example_model('child')
# cardinality of all nodes
cardinalities_child = {node: len(child_model.get_cpds(node).state_names[node]) for node in child_model.nodes()}
cardinalities_child

{'BirthAsphyxia': 2,
 'HypDistrib': 2,
 'HypoxiaInO2': 3,
 'CO2': 3,
 'ChestXray': 5,
 'Grunting': 2,
 'LVHreport': 2,
 'LowerBodyO2': 3,
 'RUQO2': 3,
 'CO2Report': 2,
 'XrayReport': 5,
 'Disease': 6,
 'GruntingReport': 2,
 'Age': 3,
 'LVH': 2,
 'DuctFlow': 3,
 'CardiacMixing': 4,
 'LungParench': 3,
 'LungFlow': 3,
 'Sick': 2}

In [8]:
alarm_model = get_example_model('alarm')
barley_model = get_example_model('barley')
child_model = get_example_model('child')
insurance_model = get_example_model('insurance')
hailfinder_model = get_example_model('hailfinder')
hepar_model = get_example_model('hepar2')
win95pts_model = get_example_model('win95pts')


In [9]:
# cardinality of all nodes in model
test_model = win95pts_model
cardinalities = {node: len(test_model.get_cpds(node).state_names[node]) for node in test_model.nodes()}
cardinalities

{'AppOK': 2,
 'DataFile': 2,
 'AppData': 2,
 'DskLocal': 2,
 'PrtSpool': 2,
 'PrtOn': 2,
 'PrtPaper': 2,
 'NetPrint': 2,
 'PrtDriver': 2,
 'PrtThread': 2,
 'EMFOK': 2,
 'GDIIN': 2,
 'DrvSet': 2,
 'DrvOK': 2,
 'GDIOUT': 2,
 'PrtSel': 2,
 'PrtDataOut': 2,
 'PrtPath': 2,
 'NtwrkCnfg': 2,
 'PTROFFLINE': 2,
 'NetOK': 2,
 'PrtCbl': 2,
 'PrtPort': 2,
 'CblPrtHrdwrOK': 2,
 'LclOK': 2,
 'DSApplctn': 2,
 'PrtMpTPth': 2,
 'DS_NTOK': 2,
 'DS_LCLOK': 2,
 'PC2PRT': 2,
 'PrtMem': 2,
 'PrtTimeOut': 2,
 'FllCrrptdBffr': 2,
 'TnrSpply': 2,
 'PrtData': 2,
 'Problem1': 2,
 'AppDtGnTm': 2,
 'PrntPrcssTm': 2,
 'DeskPrntSpd': 2,
 'PgOrnttnOK': 2,
 'PrntngArOK': 2,
 'ScrnFntNtPrntrFnt': 2,
 'CmpltPgPrntd': 2,
 'GrphcsRltdDrvrSttngs': 2,
 'EPSGrphc': 2,
 'NnPSGrphc': 2,
 'PrtPScript': 2,
 'PSGRAPHIC': 2,
 'Problem4': 2,
 'TrTypFnts': 2,
 'FntInstlltn': 2,
 'PrntrAccptsTrtyp': 2,
 'TTOK': 2,
 'NnTTOK': 2,
 'Problem5': 2,
 'LclGrbld': 2,
 'NtGrbld': 2,
 'GrbldOtpt': 2,
 'HrglssDrtnAftrPrnt': 2,
 'REPEAT': 2,
 'A

In [ ]:
# Very Large Nets
andes_model = get_example_model('andes')
link_model = get_example_model('link')
pathfinder_model = get_example_model('pathfinder')

In [10]:
child_model.name = 'child'
insurance_model.name = 'insurance'
alarm_model.name = 'alarm'
#hepar_model.name = 'hepar'
hailfinder_model.name = 'hailfinder'
#win95pts_model.name = 'win95pts'
#barley_model.name = 'barley'
#voting_model.name = 'voting'
#chess_model.name = 'chess'
#andes_model.name = 'andes'
#link_model.name = 'link'
#pathfinder_model.name = 'pathfinder'

In [16]:
# show binary variables in very large models
for model in [andes_model, link_model, pathfinder_model]:
    binary_vars = [node for node in model.nodes() if len(model.get_cpds(node).state_names[node]) == 2]
    print(f"Binary variables in {model.name} model: {binary_vars}")

Binary variables in andes model: ['GOAL_2', 'SNode_3', 'SNode_4', 'SNode_5', 'SNode_6', 'SNode_7', 'DISPLACEM0', 'RApp1', 'GIVEN_1', 'RApp2', 'SNode_8', 'SNode_9', 'SNode_10', 'SNode_11', 'SNode_12', 'SNode_13', 'SNode_14', 'SNode_15', 'SNode_16', 'SNode_17', 'SNode_18', 'SNode_19', 'NEED1', 'SNode_20', 'GRAV2', 'SNode_21', 'VALUE3', 'SNode_24', 'SLIDING4', 'SNode_25', 'CONSTANT5', 'SNode_26', 'KNOWN6', 'VELOCITY7', 'SNode_47', 'RApp3', 'KNOWN8', 'RApp4', 'SNode_27', 'COMPO16', 'GOAL_48', 'TRY12', 'TRY11', 'GOAL_49', 'CHOOSE19', 'GOAL_50', 'SYSTEM18', 'SNode_51', 'KINEMATI17', 'SNode_52', 'IDENTIFY10', 'GOAL_53', 'IDENTIFY9', 'SNode_28', 'TRY13', 'TRY14', 'TRY15', 'VAR20', 'SNode_29', 'SNode_31', 'GIVEN21', 'SNode_33', 'SNode_34', 'VECTOR27', 'APPLY32', 'GOAL_56', 'CHOOSE35', 'GOAL_57', 'MAXIMIZE34', 'SNode_59', 'AXIS33', 'SNode_60', 'WRITE31', 'GOAL_61', 'WRITE30', 'GOAL_62', 'RESOLVE37', 'GOAL_63', 'NEED36', 'SNode_64', 'SNode_41', 'SNode_42', 'IDENTIFY39', 'SNode_43', 'RESOLVE38', '

In [11]:
def select_optimal_target_node_old(bn):
    """
    Selects a target node deeply embedded in the network (highest degree).
    """
    best_node = None
    max_degree = -1
    
    for node in bn.nodes():
        # Ensure it's a binary node
        if len(bn.get_cpds(node).state_names[node]) != 2:
            continue
            
        degree = len(bn.get_parents(node)) + len(bn.get_children(node))
        if degree > max_degree:
            max_degree = degree
            best_node = node
            
    # Fallback if no binary nodes exist (rare)
    if best_node is None:
        return random.choice(list(bn.nodes()))
        
    return best_node

#antes_target = select_optimal_target_node_old(andes_model)
#link_target = select_optimal_target_node_old(link_model)
#pathfinder_target = select_optimal_target_node_old(pathfinder_model)
#print(f"Selected target node for ANDES: {antes_target}")
#print(f"Selected target node for LINK: {link_target}")
#print(f"Selected target node for PATHFINDER: {pathfinder_target}")

# Run Experiment

In [11]:
def get_target(model):
    targets = {
        'child': 'Sick',
        'alarm': 'HYPOVOLEMIA',
        'barley': 'pesticid',
        'insurance': 'Theft',
        'hailfinder': 'ScenRelAMCIN',
        'hepar': 'hepatomegaly',
        'win95pts': 'PrtMem',
        'voting': 'Class',
        'chess': 'skach',
        'andes': 'NEED36',
        'link': 'N21_d_m',
        'pathfinder': 'F97'
    }

    return targets[model.name]

## ensure all targets are present in the respective models
#for model in [child_model, alarm_model, barley_model, insurance_model, hailfinder_model, hepar_model, win95pts_model, andes_model, link_model, pathfinder_model]:
#    print(f"Checking target node for model '{model.name}'...")
#    target = get_target(model)
#    if target not in model.nodes():
#        raise ValueError(f"Target node '{target}' not found in model '{model.name}'")

def get_h_ratio(model):
    ratios = {
        'child': 0.5,
        'alarm': 0.30,
        'hepar': 0.20,
        'barley': 0.20,
        'mildew': 0.30,
        'water': 0.30,
        'hailfinder': 0.30,
        'win95pts': 0.20,
        'insurance': 0.40,
        'voting': 0.5,
        'chess': 0.9 #era 0.86
    }
    return ratios[model.name]
    



In [13]:
models_to_run = [child_model, hailfinder_model]

In [14]:
#models_to_run = [chess_model]

In [15]:
for model in models_to_run:
    print(f"Model '{model.name}' has {len(model.nodes())} variables.")

Model 'child' has 20 variables.
Model 'hailfinder' has 56 variables.


In [16]:
len(models_to_run)

2

In [17]:
all_targets_are_binary = True
for bn in models_to_run:
    #print(f"\n=== BN: {bn.name} ===")
    target = get_target(bn)
    if target is None:
        #print(f"--> No binary target defined for {bn.name}, skipping.")
        continue
    target_states = bn.get_cpds(target).state_names[target]
    if len(target_states) != 2:
        #print(f"--> Target '{target}' in {bn.name} is not binary (States: {target_states}), skipping.")
        all_targets_are_binary = False
        continue
    #print(f"Available states for target '{target}': {target_states}")
    target_value = target_states[1] if len(target_states) > 1 else target_states[0]
    #print(f"Target Node: {target}, Target Value: {target_value}")

print(f"\nAll targets are binary: {all_targets_are_binary}")


All targets are binary: True


In [18]:
for bn in models_to_run:
    print(f"\n=== BN: {bn.name} ===")
    all_nodes = list(bn.nodes())
    
    target = get_target(bn)
    if target is None:
        print(f"--> No binary target defined for {bn.name}, skipping.")
        continue
    target_states = bn.get_cpds(target).state_names[target]
    target_value = target_states[1] if len(target_states) > 1 else target_states[0]
    print(f"Target Node: {target}, Target Value: {target_value}")
    
    available_nodes = [n for n in all_nodes if n != target]
    print(f"H ratio: {get_h_ratio(bn)}")
    n_hidden = max(1, int(len(available_nodes) * get_h_ratio(bn)))
    print(f"using {n_hidden} H variables")


=== BN: child ===
Target Node: Sick, Target Value: no
H ratio: 0.5
using 9 H variables

=== BN: hailfinder ===
Target Node: ScenRelAMCIN, Target Value: CThruK
H ratio: 0.3
using 16 H variables


In [ ]:
import time
from xml.parsers.expat import model
import tracemalloc

models_to_run = [child_model, alarm_model,hailfinder_model]

def run_for_time(func, *args, **kwargs):
    """Runs natively at maximum speed to record pure execution time."""
    start_time = time.time()
    try:
        result = func(*args, **kwargs)
        return result, (time.time() - start_time), True
    except Exception as e:
        return None, np.nan, False # Failed

def run_for_memory(func, *args, **kwargs):
    """Runs with tracemalloc to record peak memory. Ignores execution time."""
    tracemalloc.start()
    try:
        func(*args, **kwargs)
    except Exception:
        pass # We just want to see how high memory got before it crashed/finished
        
    _, peak_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return peak_mem / (1024 * 1024) # Return MB

def select_random_target(bn):
    # list all binary variables in the BN
    binary_vars = []
    for node in bn.nodes():
        cpd = bn.get_cpds(node)
        if cpd is not None and len(cpd.state_names[node]) == 2:
            binary_vars.append(node)
    if not binary_vars:
        return None, None
    print(f"Binary variables in {bn.name}: {binary_vars}")
    selected_target = random.choice(binary_vars)
    return selected_target

def run_targeted_sdp_experiment(output_csv="targeted_sdp_benchmark.csv"):
    
    results = []
    raw_results = []
    #H_RATIO = 0.20
    DECISION_THRESHOLD = 0.5
    TARGET_BUCKETS = [0.5, 0.6, 0.8, 1.0]
    MCMC_TRIALS = 10 
    
    for bn in models_to_run:
        n_nodes = bn.number_of_nodes()
        print(f"\n========================================")
        print(f"Processing BN: {bn.name}")
        
        all_nodes = list(bn.nodes())
        
        target = get_target(bn)
        #target = select_random_target(bn)
        if target is None:
            print(f"--> No binary target defined for {bn.name}, skipping.")
            continue
        target_states = bn.get_cpds(target).state_names[target]
        target_value = target_states[1] if len(target_states) > 1 else target_states[0]
        print(f"Target Node: {target}, Target Value: {target_value}")
        
        available_nodes = [n for n in all_nodes if n != target]
        print(f"H ratio: {get_h_ratio(bn)}")
        #n_hidden = max(1, int(len(available_nodes) * get_h_ratio(bn)))
        
        n_hidden = 10
        
        print(f"using {n_hidden} H variables")
        n_evidence = len(available_nodes) - n_hidden
        print(f"and {n_evidence} evidence variables")
        #hidden_vars = random.sample(available_nodes, n_hidden)
        #evidence_vars = [n for n in available_nodes if n not in hidden_vars]
        
            
        harvested_data = find_exact_experimental_patients_random(bn, target, target_value, DECISION_THRESHOLD,
                                                          n_evidence, buckets=TARGET_BUCKETS, batch_size=200)
        
        # Now process whatever it managed to find
        for target_sdp, result in harvested_data.items():
            if result is None:
                continue # We didn't find a patient for this specific bucket in this network
                
            patient, exact_sdp = result
            hidden_vars = [n for n in bn.nodes() if n not in patient and n != target]
            print(f"\n  -> Benchmarking found patient for bucket {target_sdp} (Exact: {exact_sdp:.4f})")
            
            # ========================================================
            # EXACT SDP EVALUATION
            # ========================================================
            partitions = get_partitions(bn, hidden_vars, target, patient)
            print(f"       -> Running Exact SDP...")
            
            # Pass 1: Time
            exact_sdp, exact_time, exact_success = run_for_time(
                fast_broadcast_sdp, bn, target, target_value, patient, DECISION_THRESHOLD, partitions
            )
            
            # Pass 2: Memory
            exact_mem_mb = run_for_memory(
                fast_broadcast_sdp, bn, target, target_value, patient, DECISION_THRESHOLD, partitions
            )
            
            if exact_success:
                print(f"          Time: {exact_time:.4f} sec | Peak Memory: {exact_mem_mb:.2f} MB")
            else:
                print(f"          [FAILED]: Crashed at {exact_mem_mb:.2f} MB")

            # ========================================================
            # MCMC EVALUATION
            # ========================================================
            mcmc_estimates = []
            mcmc_times = []
            
            print(f"       -> Running MCMC SDP (Trials: {MCMC_TRIALS})...")
            
            # Pass 1: Pure Time (across all trials)
            for trial in range(MCMC_TRIALS):
                est_sdp, t_time, _ = run_for_time(
                    fast_mcmc_sdp_estimation_new, bn, target, target_value, patient, DECISION_THRESHOLD,
                    n_samples=1000, burn_in=2000, thinning=50
                )
                mcmc_estimates.append(est_sdp)
                mcmc_times.append(t_time)
                
            mcmc_mean = np.mean(mcmc_estimates)
            mcmc_avg_time = np.mean(mcmc_times)
            mcmc_variance = np.var(mcmc_estimates)

            # Pass 2: Peak Memory
            mcmc_mem_mb = run_for_memory(
                fast_mcmc_sdp_estimation_new, bn, target, target_value, patient, DECISION_THRESHOLD,
                n_samples=100, burn_in=50, thinning=5
            )
            
            print(f"          Avg Time: {mcmc_avg_time:.4f} sec | Peak Memory: {mcmc_mem_mb:.2f} MB")
            
            absolute_error = abs(exact_sdp - mcmc_mean)

            # ========================================================
            # PARALLEL TEMPERING MCMC EVALUATION
            # ========================================================

            #pt_mcmc_estimates = []
            #pt_mcmc_times = []
#
            #print(f"       -> Running Parallel Tempering MCMC SDP (Trials: {MCMC_TRIALS})...")              
            #
            ## Pass 1: Pure Time (across all trials)
            #for trial in range(MCMC_TRIALS):
            #    est_sdp, t_time, _ = run_for_time(
            #        pt_mcmc_sdp_estimation, bn, target, target_value, patient, DECISION_THRESHOLD,
            #        n_samples=1000, burn_in=2000, thinning=50, n_chains=4, max_temp=40.0
            #    )
            #    pt_mcmc_estimates.append(est_sdp)
            #    pt_mcmc_times.append(t_time)
#
            #pt_mcmc_mean = np.mean(pt_mcmc_estimates)
            #pt_mcmc_avg_time = np.mean(pt_mcmc_times)
            #pt_mcmc_variance = np.var(pt_mcmc_estimates)
#
            ## Pass 2: Peak Memory
            #pt_mcmc_mem_mb = run_for_memory(
            #    pt_mcmc_sdp_estimation, bn, target, target_value, patient, DECISION_THRESHOLD,
            #    n_samples=100, burn_in=50, thinning=5, n_chains=4, max_temp=10.0
            #)
#
            #print(f"          Avg Time: {pt_mcmc_avg_time:.4f} sec | Peak Memory: {pt_mcmc_mem_mb:.2f} MB")
#
            #absolute_error_pt = abs(exact_sdp - pt_mcmc_mean)
            
        
            # Record everything to the dataset
            results.append({
                'Network': bn.name,
                'N_Nodes': n_nodes,
                'Target_Bucket': target_sdp,
                'Target_Node': target,
                'Target_Value': target_value,
                'Exact_SDP': exact_sdp,
                'Exact_Time_sec': exact_time,
                'MCMC_Mean_SDP': mcmc_mean,
                'MCMC_Variance': mcmc_variance,
                'MCMC_Avg_Time_sec': mcmc_avg_time,
                'Absolute_Error': absolute_error
                #'PT_MCMC_Mean_SDP': pt_mcmc_mean,
                #'PT_MCMC_Variance': pt_mcmc_variance,
                #'PT_MCMC_Avg_Time_sec': pt_mcmc_avg_time,
                #'PT_Absolute_Error': absolute_error_pt
            })
            
            # Save progressively
            pd.DataFrame(results).to_csv(output_csv, index=False)
            pd.DataFrame(raw_results).to_csv("raw_" + output_csv, index=False)

    print(f"\nExperiment Complete! Results saved to {output_csv}")
    return pd.DataFrame(results)

In [24]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="pgmpy")
run_targeted_sdp_experiment()


Processing BN: child
Target Node: Sick, Target Value: no
H ratio: 0.5
using 10 H variables
and 9 evidence variables

Hunting for patients... (Locking 9 variables as evidence)
Generating batch 1/2 of 200 random realities...
--> Filled bucket 0.6 with Exact SDP: 0.5794
--> Filled bucket 1.0 with Exact SDP: 1.0000
--> Filled bucket 0.5 with Exact SDP: 0.5310
--> Filled bucket 0.8 with Exact SDP: 0.8105
All buckets filled successfully!

  -> Benchmarking found patient for bucket 0.5 (Exact: 0.5310)
       -> Running Exact SDP...
          Time: 0.0061 sec | Peak Memory: 0.06 MB
       -> Running MCMC SDP (Trials: 10)...
          Avg Time: 1.0512 sec | Peak Memory: 0.29 MB

  -> Benchmarking found patient for bucket 0.6 (Exact: 0.5794)
       -> Running Exact SDP...
          Time: 0.0127 sec | Peak Memory: 0.77 MB
       -> Running MCMC SDP (Trials: 10)...
          Avg Time: 1.1000 sec | Peak Memory: 0.29 MB

  -> Benchmarking found patient for bucket 0.8 (Exact: 0.8105)
       -> Runni

/home/joao/anaconda3/envs/bn-medical/lib/python3.8/site-packages/pgmpy/factors/discrete/DiscreteFactor.py:478: RuntimeWarning: invalid value encountered in divide
  phi.values = phi.values / phi.values.sum()


--> Filled bucket 0.6 with Exact SDP: 0.6427
--> Filled bucket 0.8 with Exact SDP: 0.7800
All buckets filled successfully!

  -> Benchmarking found patient for bucket 0.5 (Exact: 0.5419)
       -> Running Exact SDP...
          Time: 0.0104 sec | Peak Memory: 0.11 MB
       -> Running MCMC SDP (Trials: 10)...
          Avg Time: 1.1349 sec | Peak Memory: 0.38 MB

  -> Benchmarking found patient for bucket 0.6 (Exact: 0.6427)
       -> Running Exact SDP...
          Time: 0.0131 sec | Peak Memory: 0.48 MB
       -> Running MCMC SDP (Trials: 10)...
          Avg Time: 1.9435 sec | Peak Memory: 0.43 MB

  -> Benchmarking found patient for bucket 0.8 (Exact: 0.7800)
       -> Running Exact SDP...
          Time: 0.0101 sec | Peak Memory: 0.11 MB
       -> Running MCMC SDP (Trials: 10)...
          Avg Time: 1.3182 sec | Peak Memory: 0.38 MB

  -> Benchmarking found patient for bucket 1.0 (Exact: 1.0000)
       -> Running Exact SDP...
          Time: 0.0147 sec | Peak Memory: 0.11 MB
      

/home/joao/anaconda3/envs/bn-medical/lib/python3.8/site-packages/pgmpy/factors/discrete/DiscreteFactor.py:478: RuntimeWarning: invalid value encountered in divide
  phi.values = phi.values / phi.values.sum()


Generating batch 2/2 of 200 random realities...
Finished searching. Could not find patients for buckets: [0.5, 0.6, 0.8, 1.0]

Experiment Complete! Results saved to targeted_sdp_benchmark.csv


,Network,N_Nodes,Target_Bucket,Target_Node,Target_Value,Exact_SDP,Exact_Time_sec,MCMC_Mean_SDP,MCMC_Variance,MCMC_Avg_Time_sec,Absolute_Error
0,child,20,0.5,Sick,no,0.531043,0.006088,0.5347,0.000273,1.051176,0.003657
1,child,20,0.6,Sick,no,0.579392,0.012697,0.5793,0.000791,1.100034,0.000092
2,child,20,0.8,Sick,no,0.810535,0.012032,0.8005,0.000696,1.151947,0.010035
3,child,20,1.0,Sick,no,1.000000,0.007608,1.0000,0.000000,1.118255,0.000000
4,alarm,37,0.5,HYPOVOLEMIA,FALSE,0.541886,0.010442,0.5401,0.000233,1.134900,0.001786
5,alarm,37,0.6,HYPOVOLEMIA,FALSE,0.642670,0.013098,0.6456,0.000256,1.943502,0.002930
6,alarm,37,0.8,HYPOVOLEMIA,FALSE,0.779957,0.010068,0.7884,0.000227,1.318211,0.008443
7,alarm,37,1.0,HYPOVOLEMIA,FALSE,1.000000,0.014741,1.0000,0.000000,1.528129,0.000000


# TEST AND DEBUG

In [39]:
def teste_mcmc(bn, target, target_value, patient, threshold,
                              n_samples=11000, burn_in=1000, thinning=10):
    """
    Estimates the Same-Decision Probability via Metropolis-Hastings MCMC.

    Improvements over previous version:
      1. FIX — seed is drawn proportionally to likelihood weight instead of
         always taking the mode, preventing systematic chain trapping in
         networks where the "flip decision" region is isolated from the mode.
      2. SPEED — MH acceptance ratio is computed via a local CPD update:
         only the CPDs that involve the flipped variable are re-evaluated,
         reducing each iteration from O(all CPDs) to O(Markov blanket).
    """

    hidden_vars = [n for n in bn.nodes() if n not in patient and n != target]
    target_states = bn.get_cpds(target).state_names[target]
    evidence_states = [State(var, val) for var, val in patient.items()]

    # ── Precompute structures used on every MH step ───────────────────────────
    cpd_cache      = {n: bn.get_cpds(n) for n in bn.nodes()}
    children_cache = {v: list(bn.get_children(v)) for v in hidden_vars}

    # For each hidden var, the "affected nodes" when it is flipped are itself
    # plus its children — these are the only CPD terms whose value changes.
    affected_cache = {v: [v] + children_cache[v] for v in hidden_vars}

    # ── Seed: sample proportionally to weight (FIX) ───────────────────────────
    print("Begin sampler")
    sampler = BayesianModelSampling(bn)
    valid_seed_found = False
    while not valid_seed_found:
        print("Sampling seeds...")
        seed_df = sampler.likelihood_weighted_sample(
            size=100, evidence=evidence_states, show_progress=False
        )
        valid_seeds = seed_df[seed_df['_weight'] > 0]
        if not valid_seeds.empty:
            weights = valid_seeds['_weight'].values.astype(float)
            weights /= weights.sum()
            seed_row = valid_seeds.iloc[np.random.choice(len(valid_seeds), p=weights)]
            current_h = {v: seed_row[v] for v in hidden_vars}
            valid_seed_found = True
    print("Seed found, starting MCMC iterations...")

    # ── Helper: log P(H, E, target=t) for a full state ───────────────────────
    def full_log_joint(h_dict, t_state):
        full = {**h_dict, **patient, target: t_state}
        lp = 0.0
        for cpd in cpd_cache.values():
            p = cpd.get_value(**{v: full[v] for v in cpd.variables})
            if p == 0.0:
                return float('-inf')
            lp += math.log(p)
        return lp

    # log P(H, E) = log Σ_t P(H, E, t)  — stored as dict over target states
    def log_sum_joints(lj_dict):
        vals = [v for v in lj_dict.values() if v != float('-inf')]
        if not vals:
            return float('-inf')
        m = max(vals)
        return m + math.log(sum(math.exp(v - m) for v in vals))

    # ── Local update (SPEED): recompute only affected CPD terms ──────────────
    # When hidden var V is flipped old→new, the log joint changes by:
    #   Δ(t) = Σ_{node ∈ affected(V)} [ log P(node|pa, new) - log P(node|pa, old) ]
    # This is O(|affected(V)|) instead of O(all CPDs).
    def local_log_delta(var, old_val, new_val, h_dict, t_state):
        full_old = {**h_dict, **patient, target: t_state}
        full_new = {**full_old, var: new_val}
        delta = 0.0
        for node in affected_cache[var]:
            cpd   = cpd_cache[node]
            cvars = cpd.variables
            p_old = cpd.get_value(**{v: full_old[v] for v in cvars})
            p_new = cpd.get_value(**{v: full_new[v] for v in cvars})
            if p_new == 0.0:
                return float('-inf')
            if p_old == 0.0:
                # current joint was -inf; caller will recompute from scratch
                return float('inf')
            delta += math.log(p_new) - math.log(p_old)
        return delta

    # ── Initialise running log joints ─────────────────────────────────────────
    current_lj  = {t: full_log_joint(current_h, t) for t in target_states}
    current_log_p = log_sum_joints(current_lj)

    # ── Metropolis-Hastings loop ───────────────────────────────────────────────
    total_iters    = burn_in + n_samples * thinning
    accepted_samples = []

    for i in range(total_iters):
        var      = random.choice(hidden_vars)
        cur_val  = current_h[var]
        others   = [s for s in cpd_cache[var].state_names[var] if s != cur_val]
        if not others:
            continue
        new_val  = random.choice(others)

        # Proposed log joints via local delta (fast path)
        proposed_lj = {}
        recompute   = False
        for t in target_states:
            if current_lj[t] == float('-inf'):
                recompute = True
                break
            delta = local_log_delta(var, cur_val, new_val, current_h, t)
            if delta == float('inf'):       # old CPD was zero → need full pass
                recompute = True
                break
            proposed_lj[t] = current_lj[t] + delta

        if recompute:
            tmp_h = {**current_h, var: new_val}
            proposed_lj = {t: full_log_joint(tmp_h, t) for t in target_states}

        proposed_log_p = log_sum_joints(proposed_lj)

        # MH acceptance
        log_alpha = proposed_log_p - current_log_p
        if log_alpha >= 0 or (
            proposed_log_p != float('-inf') and
            math.log(random.random()) < log_alpha
        ):
            current_h[var]  = new_val
            current_lj      = proposed_lj
            current_log_p   = proposed_log_p

        if i >= burn_in and (i - burn_in) % thinning == 0:
            accepted_samples.append(current_h.copy())

    # ── Evaluate decision boundary ────────────────────────────────────────────
    count_same = 0
    decision_cache = {}

    for sample_h in accepted_samples:
        key = tuple(sorted(sample_h.items()))
        if key not in decision_cache:
            full_ev = {**patient, **sample_h}
            p = get_exact_target_posterior_O1(bn, target, target_value, full_ev)
            decision_cache[key] = (p >= threshold)
        if decision_cache[key]:
            count_same += 1

    return count_same / len(accepted_samples)

In [38]:
# generate random patient for hailfinder and run MCMC on it
all_nodes = list(hailfinder_model.nodes())
target = get_target(hailfinder_model)
target_states = hailfinder_model.get_cpds(target).state_names[target]
target_value = target_states[1] if len(target_states) > 1 else target_states[0]
available_nodes = [n for n in all_nodes if n != target]
n_hidden = 49
n_evidence = len(available_nodes) - n_hidden

# Randomly pick evidence variables and assign them random states
evidence_vars = random.sample(available_nodes, n_evidence)
patient = {
    var: random.choice(hailfinder_model.get_cpds(var).state_names[var])
    for var in evidence_vars
}

print(f"Target: {target} = {target_value}")
print(f"Evidence ({n_evidence} vars), hidden ({n_hidden} vars)")

# Run MCMC and time it
start = time.time()
mcmc_estimate = teste_mcmc(
    hailfinder_model, target, target_value, patient, threshold=0.5,
    n_samples=1000, burn_in=1000, thinning=50
)
elapsed = time.time() - start
print(f"MCMC SDP estimate: {mcmc_estimate:.4f} | Time: {elapsed:.2f} sec")


Target: ScenRelAMCIN = CThruK
Evidence (6 vars), hidden (49 vars)
Begin sampler


KeyboardInterrupt: 

In [69]:
all_nodes = list(win95pts_model.nodes())
win95pts_model.name = 'win95pts'
target = get_target(win95pts_model)
target_states = win95pts_model.get_cpds(target).state_names[target]
target_value = target_states[1] if len(target_states) > 1 else target_states[target]
available_nodes = [n for n in all_nodes if n != target]
n_hidden = 56
n_evidence = len(available_nodes) - n_hidden

evidence_vars = random.sample(available_nodes, n_evidence)
patient = {
    var: random.choice(win95pts_model.get_cpds(var).state_names[var])
    for var in evidence_vars}
partitions = get_partitions(win95pts_model, [n for n in win95pts_model.nodes() if n not in patient and n != target], target, patient)
max_partition_size = max(len(p) for p in partitions)
print(f"Max partition size: {max_partition_size}")
from math import prod
def compute_tensor_size(bn, partition):
    return prod(len(bn.get_cpds(v).state_names[v]) for v in partition)


def compute_max_tensor_size(bn, partitions):
    if not partitions:
        return 0
    return max(compute_tensor_size(bn, p) for p in partitions)

max_tensor_size = compute_max_tensor_size(win95pts_model, partitions)
print(f"Max tensor size (number of entries): {max_tensor_size}")
print(max_tensor_size<33_554_432)

Max partition size: 45
Max tensor size (number of entries): 35184372088832
False


In [76]:
def memory_aware_random_harvester(bn, target_node, target_value, decision_threshold,
                                            n_evidence,
                                            buckets=[0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
                                            tolerance=0.05,
                                            batch_size=500,
                                            max_batches=1,
                                            max_tensor_entries=33_554_432):
    """
    -> Same logic as function find_exact_experimental_patients_random, 
    but with an explicit memory wall check
    
    Random-evidence harvester that fills target SDP buckets with example patients.
    
    Uses tensor size (product of hidden-variable cardinalities) as the memory
    safety metric, so it works correctly on networks with mixed cardinalities
    like Child (not just binary synthetic BNs).
    
    Parameters
    ----------
    max_tensor_entries : int
        Maximum number of entries allowed in any single tensor during SDP.
    
    Returns
    -------
    dict with keys:
        'buckets'       : {bucket_value: (patient_dict, exact_sdp) | None}
        'wall_hits'     : int — patients rejected for exceeding the memory wall
        'attempts_ok'   : int — patients that passed the wall and got evaluated
        'sdp_failures'  : int — patients that passed the wall but crashed in SDP
        'inference_failures' : int — patients whose base inference crashed
    """
    all_nodes = list(bn.nodes())
    available_nodes = [n for n in all_nodes if n != target_node]

    unfilled_buckets = {b: None for b in buckets}
    wall_hits = 0
    attempts_ok = 0
    sdp_failures = 0
    inference_failures = 0

    print(f"\nHunting for patients... (Locking {n_evidence} variables as evidence)")
    print(f"Memory wall: tensor size ≤ {max_tensor_entries:,} entries "
          f"(~{max_tensor_entries * 8 / 1024**3:.1f} GB raw)")

    base_inference = VariableElimination(bn)
    batch_count = 0

    while any(v is None for v in unfilled_buckets.values()) and batch_count < max_batches:
        batch_count += 1
        print(f"Generating batch {batch_count}/{max_batches} of {batch_size} random realities...")

        for i in range(batch_size):
            # 1. Generate a random patient with randomly-sampled evidence
            print(f"  Patient {i+1}/{batch_size}...\n")
            evidence_vars = random.sample(available_nodes, min(n_evidence, len(available_nodes)))
            hidden_vars = [n for n in all_nodes if n not in evidence_vars and n != target_node]

            temp_patient = {
                var: random.choice(bn.get_cpds(var).state_names[var])
                for var in evidence_vars
            }

            # 2. Build partitions for this patient
            partitions = get_partitions(bn, hidden_vars, target_node, temp_patient)

            if not partitions:
                print(f"    [!] No partitions found for this patient, skipping.")
                continue  # nothing to compute

            # 3. MEMORY SAFETY — reject if any tensor exceeds the wall
            max_tensor = compute_max_tensor_size(bn, partitions)
            if max_tensor > max_tensor_entries:
                print(f"    [!] Rejected (tensor size {max_tensor:,} entries exceeds wall)")
                wall_hits += 1
                continue

            # 4. Check base decision meets the threshold
            try:
                base_dist = base_inference.query(
                    variables=[target_node], evidence=temp_patient, show_progress=False
                )
                if base_dist.get_value(**{target_node: target_value}) < decision_threshold:
                    continue  # legitimate rejection, not a failure
            except (ValueError, MemoryError) as e:
                inference_failures += 1
                print(f"    [!] Base inference failed ({type(e).__name__}): {e}")
                gc.collect()
                continue  # try another patient rather than bailing out

            # 5. Calculate exact SDP
            try:
                exact_sdp = fast_broadcast_sdp(
                    bn, target_node, target_value, temp_patient,
                    decision_threshold, partitions
                )
            except (ValueError, MemoryError) as e:
                sdp_failures += 1
                print(f"    [!] SDP failed despite passing wall ({type(e).__name__}): {e}")
                gc.collect()
                continue  # try another patient

            attempts_ok += 1

            # 6. Try to fit this SDP into an empty bucket
            empty_targets = [b for b, v in unfilled_buckets.items() if v is None]
            for b in empty_targets:
                if abs(exact_sdp - b) <= tolerance:
                    unfilled_buckets[b] = (temp_patient.copy(), exact_sdp)
                    print(f"--> Filled bucket {b} with Exact SDP: {exact_sdp:.4f} "
                          f"(tensor size: {max_tensor:,})")
                    break

            # 7. Early exit if all buckets filled
            if not any(v is None for v in unfilled_buckets.values()):
                break

    # Summary
    missing = [b for b, v in unfilled_buckets.items() if v is None]
    filled = len(buckets) - len(missing)

    print(f"\nHarvest summary:")
    print(f"  Buckets filled:      {filled}/{len(buckets)}")
    if missing:
        print(f"  Missing buckets:     {missing}")
    print(f"  Wall hits:           {wall_hits} (tensor too big)")
    print(f"  Attempts past wall:  {attempts_ok}")
    print(f"  SDP failures:        {sdp_failures}")
    print(f"  Inference failures:  {inference_failures}")

    # Diagnose intractable cases
    if filled == 0 and wall_hits > 0 and attempts_ok == 0:
        print(f"  -> INTRACTABLE: every random patient exceeded the memory wall")

    return {
        'buckets': unfilled_buckets,
        'wall_hits': wall_hits,
        'attempts_ok': attempts_ok,
        'sdp_failures': sdp_failures,
        'inference_failures': inference_failures,
    }

In [77]:
data = memory_aware_random_harvester(
    win95pts_model, target, target_value, decision_threshold=0.5,
    n_evidence=19, buckets=[0.5, 0.6, 0.8, 1.0], batch_size=1000, max_batches=2)


Hunting for patients... (Locking 19 variables as evidence)
Memory wall: tensor size ≤ 33,554,432 entries (~0.2 GB raw)
Generating batch 1/2 of 1000 random realities...
  Patient 1/1000...

    [!] Rejected (tensor size 35,184,372,088,832 entries exceeds wall)
  Patient 2/1000...

    [!] Rejected (tensor size 281,474,976,710,656 entries exceeds wall)
  Patient 3/1000...

    [!] Rejected (tensor size 274,877,906,944 entries exceeds wall)
  Patient 4/1000...

    [!] Rejected (tensor size 1,073,741,824 entries exceeds wall)
  Patient 5/1000...

    [!] Rejected (tensor size 2,199,023,255,552 entries exceeds wall)
  Patient 6/1000...

    [!] Rejected (tensor size 140,737,488,355,328 entries exceeds wall)
  Patient 7/1000...

    [!] Rejected (tensor size 137,438,953,472 entries exceeds wall)
  Patient 8/1000...

    [!] Rejected (tensor size 68,719,476,736 entries exceeds wall)
  Patient 9/1000...

    [!] Rejected (tensor size 17,592,186,044,416 entries exceeds wall)
  Patient 10/1000

/home/joao/anaconda3/envs/bn-medical/lib/python3.8/site-packages/pgmpy/factors/discrete/DiscreteFactor.py:478: RuntimeWarning: invalid value encountered in divide
  phi.values = phi.values / phi.values.sum()


: 

In [41]:
# Run MCMC and time it
start = time.time()
mcmc_estimate = fast_mcmc_sdp_estimation_new(
    hailfinder_model, target, target_value, patient, threshold=0.5,
    n_samples=1000, burn_in=1000, thinning=50
)
elapsed = time.time() - start
print(f"MCMC SDP estimate: {mcmc_estimate:.4f} | Time: {elapsed:.2f} sec")

MCMC SDP estimate: 1.0000 | Time: 7.66 sec


old version was stuck inside the while loop making samples. new version only calls the sampler once, so it went faster and escaped the infinite loop. take a better look at this.